# Exercise 5 - Line Guardian: solution

## Stage A - Today's system

### A1: 20 machines into PostgreSQL

In [1]:
!docker compose up -d

[+] up 0/2
 ⠋ Container postgres Starting                                              0.1s
 ⠋ Container kafka    Starting                                              0.1s

 ⠙ Container postgres Starting                                              0.2s
 ⠙ Container kafka    Starting                                              0.2s

 ⠹ Container postgres Starting                                              0.3s
 ⠹ Container kafka    Starting                                              0.3s

 ⠸ Container postgres Starting                                              0.4s
 ⠸ Container kafka    Starting                                              0.4s
[+] up 1/2

 ✔ Container kafka    Started                                               0.4s
Error response from daemon: failed to set up container networking: driver failed programming external connectivity on endpoint postgres (b7e60c5469fee0e67a01acfeda7d01de9de416aa822618499d4eb47766722122): Bind for 0.0.0.0:5432 failed: port is alr

In [2]:
# python3 guardian/baseline_detector.py

# (stopped after ~90 s)
# python3 guardian/plant.py --sink postgres --machines 20   

In [2]:
!docker exec -it postgres psql -U postgres -d factory \
  -c "SELECT count(*) FROM readings WHERE run = (SELECT max(run) FROM readings);"

 count 
-------
  2140
(1 row)


What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug postgres
    Learn more at https://docs.docker.com/go/debug-cli/


Result: all 18,180 readings are in the table, but the detector read only 15,134. So 3,041 readings (17 %) were missed, even though the detector kept up.

Because a row gets its id when it is inserted, but other sessions only see it after the commit. The gateways commit in a different order than they insert:

1. Gateway 1 inserts rows 100-114 but has not committed yet.
2. Gateway 2 inserts rows 115-129 and commits.
3. The detector reads id > 99, sees only 115-129 and remembers 129 as its position.
4. Gateway 1 commits. Rows 100-114 are now visible, but they are below 129, so the detector never reads them.

So "the highest id I have seen" is not a safe position.

### A2: full plant, 60 machines (600 readings/s)

In [4]:
# python3 baseline_detector.py      
# (~60 s)
# # python3 plant.py --sink postgres   

- The backlog grows (777 to 20,430 rows) and the delay grows (1.4 s to 34.3 s). The model needs 3 ms per reading, so one detector can do about 333 readings/s, but 600 arrive. It falls further behind every second. R3 (5 s) is broken.
- The fault on M-029 at 40 s was never reported. The detector was still 34 s behind when we stopped.
- The missing count stops at 89. Because the detector is so far behind, everything it reads is already committed, so it no longer skips rows. The bug is hidden, not fixed.

So the old system is either incomplete (when it keeps up) or late (when it does not).

## Stage B - Decision record

600 readings/s in the lab, 100,000 readings/s at rollout (2,000 machines x 50 Hz). The model keeps state per machine and needs 3 ms per reading.

| Requirement | Option 1: better polling of PostgreSQL | Option 2: Kafka |
|---|---|---|
| R1 scale | 100,000 inserts/s plus many pollers on one database. | More partitions and more detector instances. |
| R2 complete, in order | The skipped rows problem from A1 stays, unless every poller tracks each machine separately. | Key = machine, so each machine stays in one partition and in order. Offsets have no gaps. |
| R3 fresh | Depends on the poll interval and gets worse as the table grows. |Measured p95 latency 0.08 s. |
| R4 more consumers | Every new consumer is another poller and adds load to PostgreSQL. | Each consumer gets its own group and reads independently. |

Decision: Kafka. It meets all four requirements. Polling breaks R4 by design.

The number of partitions has to be chosen in advance. When a machine moves to another detector instance, the model needs about 15 s to warm up again for that machine. After a crash a few alerts can be sent twice (they can be recognized by anomaly_id).

Topic readings with 6 partitions, key machine_id. The detectors share the group line-guardian. Alerts go to the topic anomalies, also keyed by machine_id.

How many detector instances:

In [5]:
cost_s = 0.003                                     # model.score() per reading
per_instance = 1 / cost_s                          # max readings/s of one instance
for name, rate in [("lab", 60 * 10), ("rollout", 2000 * 50)]:
    work = rate * cost_s                           # CPU-seconds of model work per second
    print(f"{name:8} {rate:>7,} readings/s -> {work:6.1f} instances at 100% load, "
          f"{int(-(-work // 0.7)):4} at 70% load")
print(f"one instance: max {per_instance:.0f} readings/s")

lab          600 readings/s ->    1.8 instances at 100% load,    3 at 70% load
rollout  100,000 readings/s ->  300.0 instances at 100% load,  429 at 70% load
one instance: max 333 readings/s


- Lab: 2 instances would be 90 % busy, too tight. We use 3 (about 60 % busy). With 6 partitions we can grow to 6 instances later.
- Rollout: about 430 instances, so the topic needs at least that many partitions (for example 500), on several brokers. A faster model would reduce this a lot.

Tipping point: at one reading per minute, PostgreSQL is fine and simpler. Kafka pays off when one consumer can no longer keep up (here about 300 readings/s), when several consumers need the same data, or at thousands of writes per second.

## Stage C - Build with Kafka

### Topics

The topics are created before the plant starts, because automatic creation would give only 1 partition. The errors "already exists" only mean the topics were created earlier.

In [6]:
!docker exec -it kafka kafka-topics.sh --bootstrap-server localhost:9092 \
  --create --topic readings --partitions 6 --replication-factor 1
!docker exec -it kafka kafka-topics.sh --bootstrap-server localhost:9092 \
  --create --topic anomalies --partitions 3 --replication-factor 1
!docker exec -it kafka kafka-topics.sh --bootstrap-server localhost:9092 --describe --topic readings

Error while executing topic command : Topic 'readings' already exists.
[2026-11-09 09:29:13,538] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'readings' already exists.
 (org.apache.kafka.tools.TopicCommand)

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug kafka
    Learn more at https://docs.docker.com/go/debug-cli/
Error while executing topic command : Topic 'anomalies' already exists.
[2026-11-09 09:29:14,676] ERROR org.apache.kafka.common.errors.TopicExistsException: Topic 'anomalies' already exists.
 (org.apache.kafka.tools.TopicCommand)

What's next:
    Try Docker Debug for seamless, persistent debugging tools in any container or image → docker debug kafka
    Learn more at https://docs.docker.com/go/debug-cli/
Topic: readings	TopicId: S-iIwRGcSl6rqfwebl4Peg	PartitionCount: 6	ReplicationFactor: 1	Configs: min.insync.replicas=1
	Topic: readings	Partition: 0	Leader: 1	Replicas: 1	Isr: 1	Elr: 	L

### KafkaSink in plant.py

```python
self.producer = Producer({
    "bootstrap.servers": args.bootstrap,
    "partitioner": "murmur2_random",   # same key -> same partition as the Kafka tools
    "acks": "all",                     # a reading only counts as sent once Kafka stored it
    "enable.idempotence": True,        # retries create no duplicates and keep the order
    "linger.ms": 10,                   # send in small batches
})

def send(self, reading):
    self.producer.produce("readings", key=reading["machine_id"], value=json.dumps(reading))
```

### Decisions in detector_service.py

- Group: all instances use the same group, line-guardian. Kafka then gives each partition (and so each machine) to exactly one instance.
- Start position: latest. A new group starts with live data instead of replaying days of old readings.
- Rebalance: when an instance loses partitions, it saves its position and the model forgets those machines. Otherwise old state would be used if the partition comes back later.
- Alerts: sent to anomalies with key machine_id, so the alerts of one machine stay in order.

### Run with 3 detector instances

In [3]:
# python3 ../guardian/scoreboard.py --reset
# python3 detector_service.py --name det-1    # terminal 1
# python3 detector_service.py --name det-2    # terminal 2
# python3 detector_service.py --name det-3    # terminal 3
# python3 plant.py --sink kafka               # terminal 4, ~3 min
!python3 ../guardian/scoreboard.py

Run 1794476285: started 10:38:05 (6.0 min ago), 60 machines x 10 Hz, 6 faults so far

Detection  (topic anomalies: 2,079 anomaly events, models ['v1'])
   #  machine   start  detected   delay
   1  M-029       40s       yes   7.9 s
   2  M-026      100s       yes   8.2 s
   3  M-031      160s       yes   6.0 s
   4  M-036      220s       yes   7.4 s
   5  M-004      280s       yes   7.6 s
   6  M-052      340s  (just started, not scored yet)

  faults detected: 5/5, mean delay 7.4 s; false alarms (episodes on healthy machines): 0
  latency reading -> anomaly event: p50 0.04 s, p95 0.07 s, max 0.09 s
  duplicate anomaly events (same anomaly_id): 0

Detector instances  (model telemetry, this run)
  det-1          pid 83987   v1  readings    86,413  missing       0  out-of-order      0  machines   25  alive
  det-2          pid 87200   v1  readings    69,137  missing       0  out-of-order      0  machines   20  alive
  det-3          pid 89265   v1  readings    51,861  missing       0  ou

Everything passes.

- No reading is missing or out of order. The three instances together read 45,175 + 36,140 + 27,105 = 108,420 readings, exactly what the plant sent.
- Alerts arrive after 0.08 s (p95), far below 5 s. The 7 to 8 s until a fault is detected come from the model, which needs some readings to see the vibration rise.
- The instances have 25, 20 and 15 machines, because the hash does not spread the machines evenly over the partitions. The busiest instance is about 75 % busy, which is fine.
- "silent for 28 s" only means the detectors were already stopped when the scoreboard ran.

In [4]:
!docker exec -it kafka kafka-consumer-groups.sh --bootstrap-server localhost:9092 \
  --describe --group line-guardian


Consumer group 'line-guardian' has no active members.

GROUP           TOPIC           PARTITION  CURRENT-OFFSET  LOG-END-OFFSET  LAG             CONSUMER-ID     HOST            CLIENT-ID
line-guardian   readings        4          27816           27816           0               -               -               -
line-guardian   readings        3          31293           31293           0               -               -               -
line-guardian   readings        2          38247           38247           0               -               -               -
line-guardian   readings        1          62586           62586           0               -               -               -
line-guardian   readings        5          24339           24339           0               -               -               -
line-guardian   readings        0          24339           24339           0               -               -               -

What's next:
    Try Docker Debug for seamless, persistent d

LAG is 0 on every partition, so the group has read everything. The offsets include all earlier runs, not only this one.

### Comparison: only one instance

Same setup, but a single detector in a new group:

In [9]:
# python3 detector_service.py --name solo-1 --group solo-test
# python3 plant.py --sink kafka
!python3 ../guardian/scoreboard.py

Run 1794214654: started 09:57:34 (31.7 min ago), 60 machines x 10 Hz, 2 faults so far

Detection  (topic anomalies: 23 anomaly events, models ['v1'])
   #  machine   start  detected   delay
   1  M-029       40s       yes   7.9 s
   2  M-005      100s        NO       -

  faults detected: 1/2, mean delay 7.9 s; false alarms (episodes on healthy machines): 0
  latency reading -> anomaly event: p50 55.98 s, p95 56.87 s, max 56.96 s
  duplicate anomaly events (same anomaly_id): 0

Detector instances  (model telemetry, this run)
  none reported

Requirements (measurable part)
  [FAIL] R2 complete and ordered: 0 readings missing, 0 out of order
  [FAIL] R3 fresh: p95 latency < 5 s
  [FAIL] detection: all faults found, at most 2 false alarms


- Still nothing missing and nothing out of order. Kafka keeps that, no matter how many instances run.
- But one instance scores only about 280 readings/s while 600 arrive. It read only 30,101 of 60,960 readings, so alerts came about 56 s late (R3 fails).
- The fault on M-005 was never found, because the detector had not reached those readings yet.
- The fix is more instances in the same group, as in the run with 3 instances.

Why a new group here: a detector in an existing group continues where that group stopped. If readings were written while no detector was running, it first reads all of those old readings before it gets to live data.